# ViFinQA — Qwen2.5-Coder-7B Select, Payload w=0.10

**Kaggle settings:** Accelerator = GPU T4 x2, Internet = On.

Notebook này dùng payload bạn đã upload tại:

```text
/kaggle/input/datasets/kien2005/kaggle-payload-w010
```

Output chính cần tải về sau khi chạy xong:

```text
/kaggle/working/codegen_sel7b_w010.jsonl
```


In [ ]:
import glob, json, pathlib

EXPLICIT_ROOT = pathlib.Path("/kaggle/input/datasets/kien2005/kaggle-payload-w010")

hits = []
if EXPLICIT_ROOT.exists():
    hits = sorted(EXPLICIT_ROOT.glob("**/retrieval.jsonl"))

if not hits:
    all_hits = [pathlib.Path(p) for p in glob.glob("/kaggle/input/**/retrieval.jsonl", recursive=True)]
    hits = [p for p in all_hits if "kaggle-payload-w010" in str(p) or "payload-w010" in str(p)]
    if not hits:
        hits = all_hits

assert hits, "Chưa attach dataset payload w010 hoặc Kaggle chưa mount input."
assert len(hits) == 1, f"Có nhiều retrieval.jsonl; hãy detach payload cũ hoặc chỉ rõ path: {hits}"

PAYLOAD = str(hits[0].parent)
manifest_path = pathlib.Path(PAYLOAD) / "payload-manifest.json"
assert manifest_path.exists(), f"Payload thiếu manifest: {manifest_path}"

manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
assert manifest.get("schema_version") == 2, f"Payload schema cũ: {manifest.get('schema_version')}"

print("PAYLOAD =", PAYLOAD)
print("manifest files =", len(manifest.get("files", {})))

import torch
print("GPUs:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("GPU 0:", torch.cuda.get_device_name(0))


In [ ]:
# Copy code da duoc fingerprint trong payload sang /kaggle/working.
import pathlib, shutil

SRC = pathlib.Path(PAYLOAD) / "code"
DST = pathlib.Path("/kaggle/working/code")
assert SRC.exists(), f"Payload thieu code/: {SRC}"

shutil.rmtree(DST, ignore_errors=True)
shutil.copytree(SRC, DST)
print("code ->", DST)


In [ ]:
%%time
# Giu trong major version da kiem tra; tranh update len major moi giua cac lan chay.
!pip install -q "transformers>=4.45,<5" "accelerate>=1,<2" "bitsandbytes>=0.45,<1"

import transformers, bitsandbytes
print("transformers", transformers.__version__)
print("bitsandbytes", bitsandbytes.__version__)


In [ ]:
%%time
# Smoke test nho de kiem payload/model/runtime truoc khi dot full GPU.
!python /kaggle/working/code/kaggle_codegen.py --payload $PAYLOAD --backend hf \
    --model Qwen/Qwen2.5-Coder-7B-Instruct --load-4bit \
    --llm-mode select --llm-target all \
    --out /kaggle/working/codegen_smoke_w010.jsonl --limit 12 \
    --n 1 --temperature 0 --k 4 --max-tokens 96 --batch-size 4 \
    --checkpoint-every 4 --time-budget-min 30 --seed 13


In [ ]:
import collections, json, pathlib

smoke = pathlib.Path("/kaggle/working/codegen_smoke_w010.jsonl")
rows = [json.loads(line) for line in smoke.open(encoding="utf-8")]
print("rows", len(rows), "unique ids", len({r["id"] for r in rows}))
print(collections.Counter(r.get("source") for r in rows))
print(rows[-1])


Nếu smoke pass, chạy full cell dưới. Cấu hình giữ giống run 7B select trước đó để so sánh công bằng: chỉ đổi retrieval/payload sang `w=0.10`.

In [ ]:
%%time
# Full run. Chay lai cung cell + cung output/config se resume theo run_signature.
!python /kaggle/working/code/kaggle_codegen.py --payload $PAYLOAD --backend hf \
    --model Qwen/Qwen2.5-Coder-7B-Instruct --load-4bit \
    --llm-mode select --llm-target all \
    --out /kaggle/working/codegen_sel7b_w010.jsonl \
    --n 1 --k 4 --max-tokens 96 --batch-size 8 \
    --checkpoint-every 32 --time-budget-min 400 --seed 13


In [ ]:
# QA toi thieu truoc khi download.
import collections, json, math, pathlib

out = pathlib.Path("/kaggle/working/codegen_sel7b_w010.jsonl")
rows = [json.loads(line) for line in out.open(encoding="utf-8")]
ids = [r["id"] for r in rows]

assert len(rows) == 1012 and len(set(ids)) == 1012, (len(rows), len(set(ids)))
assert all(math.isfinite(float(r.get("answer", 0.0))) for r in rows)

print(collections.Counter(r.get("source") for r in rows))
print("OK: 1012 unique finite results ->", out)


## Sau Khi Chạy Xong

Tải file này về local:

```text
/kaggle/working/codegen_sel7b_w010.jsonl
```

Sau đó build submission local:

```bash
python scripts/05_build_submission.py \
  --retrieval artifacts/retrieval_p1_rowrerank_full_w010.jsonl \
  --codegen <duong_dan>/codegen_sel7b_w010.jsonl \
  --out-dir artifacts/submission_sel7b_w010_codegen_k5
```

Nếu Kaggle hết phiên, tải checkpoint `codegen_sel7b_w010.jsonl` về. Ở phiên mới, upload/copy lại đúng đường dẫn `/kaggle/working/codegen_sel7b_w010.jsonl`, rồi chạy lại full cell với cùng tham số để resume.